In [1]:
!pip install openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.3/251.3 kB 2.4 MB/s eta 0:00:0000:01


In [2]:
!pip install anndata==0.8.0

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 12.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from sklearn.neighbors import KernelDensity
import time
import dill
import pickle
from scipy.spatial import distance
import os
from multiprocessing import Pool
from joblib import Parallel, delayed
import itertools

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [43]:
with open('../../parent_dict_v2.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [57]:
fn = '../../Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad'

In [58]:
sam=SAM()
sam.load_data(fn)
gene_dict = {}
for i in range(len(sam.adata.var_names)):
    gene_dict[sam.adata.var_names[i]] = i

In [59]:
sam.adata.obs.columns

Index(['n_genes', 'n_counts', 'key', 'leiden_clusters_neuron', 'eq_subclass',
       'eq_subclass_lc', 'leiden_clusters', 'eq_subclass_nounlabeled',
       'ss_subclass', 'ss_subclass_v4_nounlabeled',
       'ss_subclass_v4_nounlabeled_nn', 'ss_subclass_nounlabeled_nmm_v4_nn',
       'ss_subclass_nounlabeled_nmm_v4_nn_thresh30',
       'ss_subclass_nounlabeled_nmm_cl_v4_nn', 'neurotransmitter_v3'],
      dtype='object')

In [60]:
level = 'ss_subclass_nounlabeled_nmm_cl_v4_nn'

In [61]:
for item in sam.adata.obs[level].unique():
    if item[:2] == 'mo':
        parent_dict[item] = 'hypo'
    if item[:2] == 'mg':
        parent_dict[item] = 'hypo'
    if item[:2] == 'xt':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ri':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ac':
        parent_dict[item] = 'hypo'
    if item[:2] == 'cj':
        parent_dict[item] = 'hypo'
    if item[:2] == 'dr':
        parent_dict[item] = 'hypo'
    if item[:2] == 'mv':
        parent_dict[item] = 'hypo'
    if item[:2] == 'cc':
        parent_dict[item] = 'hypo'
    if item[:2] == 'rv':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ri':
        parent_dict[item] = 'hypo'
    if item[-2:] == 'NN':
        parent_dict[item] = 'Non neuron'

In [62]:
parent_dict['not hypo'] = 'not hypo'
parent_dict['hypo'] = 'hypo'
parent_dict['Unknown'] = 'hypo'
parent_dict['Unknwon_2'] = 'hypo'
parent_dict['Non neuron'] = 'not hypo'

In [63]:
ordered_genes = sam.identify_marker_genes_ratio(level)

In [64]:
ordered_genes.keys()

dict_keys(['041 OB-in Frmd7 Gaba', '045 OB-STR-CTX Inh IMN', '049 Lamp5 Gaba', '054 STR Prox1 Lhx6 Gaba', '056 Sst Chodl Gaba', '060 OT D3 Folh1 Gaba', '062 STR D2 Gaba', '075 MEA-BST Lhx6 Nr2e1 Gaba', '077 CEA-BST Gal Avp Gaba', '086 MPO-ADP Lhx8 Gaba', '091 ARH-PVi Six6 Dopa-Gaba', '092 TMv-PMv Tbx3 Hist-Gaba', '105 TMd-DMH Foxd2 Gaba', '111 TRS-BAC Sln Glut', '123 DMH Nkx2-4 Glut', '130 LHA Pmch Glut', '131 LHA-AHN-PVH Otp Trh Glut', '133 PVH-SO-PVa Otp Glut', '145 MH Tac2 Glut', '314 CB Granule Glut', '323 Ependymal NN', '325 CHOR NN', '326 OPC NN', '327 Oligo NN', '328 OEC NN', '329 ABC NN', '330 VLMC NN', '331 Peri NN', '333 Endo NN', '337 DC NN', '338 Lymphoid NN', '339 Astrocyte-like NN', '340 Macrophage NN', 'Unlabeled', 'ac_dr_1', 'cj_ac_xt_dr_1', 'cj_ac_xt_dr_3', 'cj_ac_xt_dr_4', 'cj_xt_dr_1', 'cj_xt_dr_2', 'dr_10', 'dr_11', 'dr_14', 'dr_16', 'dr_18', 'dr_19', 'dr_2', 'dr_20', 'dr_22', 'dr_23', 'dr_25', 'dr_26', 'dr_28', 'dr_31', 'dr_34', 'dr_35', 'dr_37', 'dr_5', 'dr_7', 'd

In [65]:
hypo_ct = [i for i in sam.adata.obs[level].unique() if parent_dict[parent_dict[i]] == 'hypo']

In [66]:
np.sort(hypo_ct)

array(['075 MEA-BST Lhx6 Nr2e1 Gaba', '077 CEA-BST Gal Avp Gaba',
       '086 MPO-ADP Lhx8 Gaba', '091 ARH-PVi Six6 Dopa-Gaba',
       '092 TMv-PMv Tbx3 Hist-Gaba', '105 TMd-DMH Foxd2 Gaba',
       '111 TRS-BAC Sln Glut', '123 DMH Nkx2-4 Glut', '130 LHA Pmch Glut',
       '131 LHA-AHN-PVH Otp Trh Glut', '133 PVH-SO-PVa Otp Glut',
       'ac_dr_1', 'cj_ac_xt_dr_1', 'cj_ac_xt_dr_3', 'cj_ac_xt_dr_4',
       'cj_xt_dr_1', 'cj_xt_dr_2', 'dr_10', 'dr_11', 'dr_14', 'dr_16',
       'dr_18', 'dr_19', 'dr_2', 'dr_20', 'dr_22', 'dr_23', 'dr_25',
       'dr_26', 'dr_28', 'dr_31', 'dr_34', 'dr_35', 'dr_37', 'dr_5',
       'dr_7', 'dr_8', 'dr_9'], dtype='<U28')

In [67]:
def markers_celltype_neighbor(cto, neigh_cto, test_top, diff, qdiffth, qdiffth2,
                              num_ct_better, A, obs_level, ordered_genes, gene_dict):
    mask = obs_level == cto
    ct_X = A[mask, :]
    bkgd_all_X = A[np.isin(obs_level, ref) & ~mask,:]                           # all cells NOT in cto
    neigh_X = {ctt: A[obs_level == ctt, :] for ctt in neigh_cto}  # precompute once

    markers = []
    for gene in ordered_genes[cto][:test_top]:
        gene_index = gene_dict[gene]
        ct_exp_g = ct_X[:, gene_index]
        fct_exp_g = np.sum(ct_exp_g > 0) / len(ct_exp_g)
        ct_mean = np.average(ct_exp_g)

        num = 0
        sig = None  # significance vs full background, computed at most once per gene
        for ctt in neigh_cto:
            bkgd_exp_g = neigh_X[ctt][:, gene_index]
            fbkgd_exp_g = np.sum(bkgd_exp_g > 0) / len(bkgd_exp_g)

            # effect size is still measured relative to the close neighbor
            crit = ((fct_exp_g > diff and (fct_exp_g - fbkgd_exp_g) / fct_exp_g > qdiffth)
                    or (ct_mean - np.average(bkgd_exp_g) > qdiffth2))
            if crit:
                if sig is None:
                    st, pval = stats.mannwhitneyu(x=ct_exp_g,
                                                  y=bkgd_all_X[:, gene_index],
                                                  alternative='greater')
                    sig = pval < .05 / test_top
                if sig:
                    num += 1

        if num >= num_ct_better:
            markers.append(gene)
    return cto, markers

In [68]:
test_top = 7000
diff = .01
qdiffth = .8
qdiffth2 = 1
kn = 10
num_ct_better = 1
query = hypo_ct
ref = list(hypo_ct)

A = sam.adata.X.A
obs_level = sam.adata.obs[level].values
pca = sam.adata.obsm['X_pca']

# --- precompute nearest-neighbor cell types once (centroids in PCA space) ---
t0 = time.time()
centroids = np.vstack([pca[obs_level == ct].mean(axis=0) for ct in ref])
neigh = {}
for q in query:
    qc = pca[obs_level == q].mean(axis=0)
    d = np.linalg.norm(centroids - qc, axis=1)
    di = np.argsort(d)[:kn + 1][1:]          # drop self (nearest, distance 0)
    neigh[q] = [ref[i] for i in di]
print('created neighbors in: ' + str(time.time() - t0) + ' seconds')

# --- parallel marker calling ---
t1 = time.time()
results = Parallel(n_jobs=8, verbose=10)(
    delayed(markers_celltype_neighbor)(cto,
                                       neigh[cto],
                                       test_top=test_top,
                                       diff=diff,
                                       qdiffth=qdiffth,
                                       qdiffth2=qdiffth2,
                                       num_ct_better=num_ct_better,
                                       A=A,
                                       obs_level=obs_level,
                                       ordered_genes=ordered_genes,
                                       gene_dict=gene_dict)
    for cto in query
)
marker_dict = dict(results)
print('called markers in: ' + str(time.time() - t1) + ' seconds')
for cto in query:
    print(cto, len(marker_dict[cto]))

created neighbors in: 0.043154001235961914 seconds


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   30.8s
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed:  1.4min
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed:  2.6min
[Parallel(n_jobs=8)]: Done  27 out of  38 | elapsed:  4.0min remaining:  1.6min
[Parallel(n_jobs=8)]: Done  31 out of  38 | elapsed:  4.7min remaining:  1.1min
[Parallel(n_jobs=8)]: Done  35 out of  38 | elapsed:  5.3min remaining:   27.5s
[Parallel(n_jobs=8)]: Done  38 out of  38 | elapsed:  6.1min finished


called markers in: 366.16722559928894 seconds
dr_9 451
dr_5 567
cj_ac_xt_dr_4 85
111 TRS-BAC Sln Glut 473
dr_7 532
cj_xt_dr_1 2532
dr_20 139
cj_ac_xt_dr_3 86
dr_18 80
dr_11 2028
086 MPO-ADP Lhx8 Gaba 65
cj_ac_xt_dr_1 3806
105 TMd-DMH Foxd2 Gaba 1535
dr_16 356
cj_xt_dr_2 123
dr_2 151
ac_dr_1 706
091 ARH-PVi Six6 Dopa-Gaba 1475
dr_23 73
dr_14 5087
dr_8 157
dr_26 963
077 CEA-BST Gal Avp Gaba 179
dr_10 161
dr_28 843
dr_19 1268
dr_35 368
092 TMv-PMv Tbx3 Hist-Gaba 84
dr_34 241
dr_31 667
123 DMH Nkx2-4 Glut 673
131 LHA-AHN-PVH Otp Trh Glut 92
dr_25 338
dr_22 322
075 MEA-BST Lhx6 Nr2e1 Gaba 765
133 PVH-SO-PVa Otp Glut 115
130 LHA Pmch Glut 29
dr_37 939


In [69]:
folder_name = 'DR_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07272026'
if not os.path.isdir('../../Important_genes/' + folder_name):
    os.mkdir('../../Important_genes/' + folder_name)
df = pd.DataFrame(data = [qdiffth,qdiffth2, diff, kn, num_ct_better], index = ['qdiffth','qdiffth2', 'diff', 'kn', 'num_ct_better']) 
df.to_csv('../../Important_genes/' + folder_name + '/metadata.csv')
with open('../../Important_genes/' + folder_name + '/' + folder_name + '.pkl', 'wb') as f:
    pickle.dump(marker_dict, f)